## HW 5: LSTM Language Models
### COSC 426: Fall 2025, Colgate University

Use this notebook to run your experiments and answer questions. Add as many code or markdown chunks as you would like for each (sub)section. Please use markdown chunks for written responses. 

**If you use any external resources (e.g., code snippets, reference articles), please cite them in comments or text!**

## Part 1: Understanding and completing the four classes

In [3]:
import LM
import torch

### Part 1.1

In [6]:
sample_data = LM.LM_Dataset("data/yoda_train.tsv", "data/HW5_vocab.txt", 5)

# Sanity check for dataset
for i in range(3):
    input_seq, target_seq = sample_data[i]
    print("Input Sequence: ", input_seq)
    print("Target Sequence:", target_seq)
    print()

Input Sequence:  tensor([ 0., 10.,  6.,  2.,  6.])
Target Sequence: tensor([ 0,  6,  2,  6, 12])

Input Sequence:  tensor([ 0., 10.,  1., 19., 23.])
Target Sequence: tensor([ 0,  1, 19, 23, 21])

Input Sequence:  tensor([ 0., 10.,  6., 12.,  6.])
Target Sequence: tensor([ 0,  6, 12,  6, 18])



The function prepares data by creating (context, target) pairs designed for next-word prediction. For each full sequence, it defines the context as the entire sequence except the last token and the target as the sequence shifted one step over. This pairing creates a one-to-one mapping where every token in the context is used to predict the token immediately following it in the original sequence. Finally, both the context and target tensors are left-padded to make them all a uniform max_length for batch processing.

The main difference is their goal: Here, the model tries to predict the next word in a sequence, given all the previous words. CBOW tries to predict a center word, given its surrounding words, both before and after. The language model function iterates only once per sequence to generate a single (context, target) pair representing the entire shifted sequence, which is then left-padded to max_length. In contrast, the CBOW function uses a nested loop to iterate over every single word in the sequence, making each word a target. For each target, it dynamically constructs a context by slicing a window of neighbors, manually calculating and inserting [PAD] tokens within that window to keep its size fixed, and concatenating the left/right sides, thus generating many pairs.

### Part 1.2

1. The decoder is a classifier. It takes the LSTM's final feature representation and converts it into a prediction. The `seqLength` isn't specified in the torch.nn.Linear definition because the layer is applied independently to every time step. PyTorch takes the input tensor and applies the same Linear transformation to all vectors, resulting in an output tensor.

2. The LSTM layer returns three things:
* `output`: a tensor containing the hidden state from the final LSTM layer for every single time step in the sequence. This is what we use to make a prediction at each point in the sequence.
* `hidden`: this is the final hidden state after the very last time step. 
* `cell`: this is the final cell state after the very last time step. 

They are important because the hidden and cell states are the memory of the LSTM. They encapsulate everything the model has learned from processing the entire sequence. In a training loop, these final states are detached and passed as the initial states for the next batch.

3. The `init_hidden` creates the initial memory for the LSTM. An LSTM needs a hidden state and a cell state to begin its calculations. This function creates two tensors of the correct shape (nLayers, batchSize, nHidden) filled entirely with zeros to provide empty memory for the model before it starts processing. It's called at the beginning of each epoch.

4. There is no difference in the loss function. Both models use `torch.nn.CrossEntropyLoss`. Both models are performing a classification task so the loss function's job is to measure how well the model's vocabSize-dimensional logit vector correctly predicts the single target.

### Part 1.3

1. 
* Initializing Hidden States:

```python
hidden, cell = model.init_hidden(X.size(0))
```

The LM_Trainer is training an LSTM. RNNs are stateful: they maintain a memory that gets updated as they process a sequence. This init_hidden call creates a blank tensor for this memory at the beginning of each new batch. The CBOW model is feed-forward, so it doesn't require this initialization.

* Reshaping Outputs：

```python
y_pred_reshaped = y_pred.reshape(-1, model.vocabSize)

y_target.flatten().long()
```

The LSTM model outputs predictions for every word in the sequence, resulting in a 3D tensor y_pred. The target y_target is a 2D tensor. However, the CrossEntropyLoss function expects a 2D input and a 1D target. These reshape and flatten operations transform the tensors to match what the loss function expects, effectively treating the entire batch of sequences as one set of batchSize * seqLength predictions. The CBOW model only predicts one word per input, so its output is already in the correct 2D format, and its target is already 1D.

2. The additional step is gradient clipping:
```python
torch.nn.utils.clip_grad_norm_(model.parameters(), 0.25)
```
Recurrent networks are trained by backpropagation. During this process, the exploding gradients might happen. When gradients explode, the model parameters update too violently, causing learning to fail. Thus this step prevents this by enforcing a maximum allowed norm for the gradients. If gradients exceed the value, they are rescaled.

### Part 1.4

1. The LM model initializes and passes hidden states. Also, it reshapes predictions and targets to compute loss across all time steps.

2. In CBOW, each input corresponds to one target word, so choosing the word with the highest logit gives the predicted center word. In language modeling, however, the model should predict the probability distribution over the vocabulary at every time step in the sentence. Instead of choosing the most likely word, we actually want the probability assigned to the correct next word. Therefore, we apply softmax to get probabilities and then extract the probability of each true next word.

## Part 2: Generating predictions from models trained on SAE and Yoda sentences

In [2]:
nEmbed = 15
nHidden = 30
nLayers = 1

In [20]:
train_data_sae = LM.LM_Dataset("data/sae_train.tsv", "data/HW5_vocab.txt", 20)
val_data_sae = LM.LM_Dataset("data/sae_val.tsv", "data/HW5_vocab.txt", 20)

model_sae = LM.LSTM_LM(train_data_sae.vocabSize, nEmbed, nHidden, nLayers)
trainer = LM.LM_Trainer(num_epochs=50, lr=0.5, batch_size=16, device="cpu")

trainer.train(model_sae, train_data_sae, val_data_sae)

Epoch 0:	 Avg Train Loss: 0.91306	 Avg Val Loss: 0.50125
Epoch 10:	 Avg Train Loss: 0.38114	 Avg Val Loss: 0.37418
Epoch 20:	 Avg Train Loss: 0.36498	 Avg Val Loss: 0.351
Epoch 30:	 Avg Train Loss: 0.33013	 Avg Val Loss: 0.3189
Epoch 40:	 Avg Train Loss: 0.29889	 Avg Val Loss: 0.2884


In [10]:
train_data_yoda = LM.LM_Dataset("data/yoda_train.tsv", "data/HW5_vocab.txt", 20)
val_data_yoda   = LM.LM_Dataset("data/yoda_val.tsv", "data/HW5_vocab.txt", 20)

model_yoda = LM.LSTM_LM(train_data_yoda.vocabSize, nEmbed, nHidden, nLayers)
trainer = LM.LM_Trainer(num_epochs=50, lr=0.5, batch_size=16, device="cpu")

trainer.train(model_yoda, train_data_yoda, val_data_yoda)

Epoch 0:	 Avg Train Loss: 0.90768	 Avg Val Loss: 0.48531
Epoch 10:	 Avg Train Loss: 0.38367	 Avg Val Loss: 0.35401
Epoch 20:	 Avg Train Loss: 0.36849	 Avg Val Loss: 0.32345
Epoch 30:	 Avg Train Loss: 0.32387	 Avg Val Loss: 0.28037
Epoch 40:	 Avg Train Loss: 0.29893	 Avg Val Loss: 0.26413


In [21]:
eval_data = LM.LM_Dataset("data/eval.tsv", "data/HW5_vocab.txt", 20)

evaluator = LM.LM_Evaluator(eval_data, device="cpu")

models = {
    "SAE": model_sae,
    "Yoda": model_yoda
}

evaluator.save_preds(models, "lm_eval_output.tsv")

## Part 3: Analyzing and interpreting the predictions with the analyze mode from NLPScholar

1. I focused on the `by_pair` output because it directly compares surprisal between the expected and unexpected sentence. This is the clearest way to see whether the model prefers the word order pattern from the variety it was trained on.

In [33]:
import pandas as pd
import matplotlib.pyplot as plt

bypair = pd.read_csv("lm_eval_results_bypair.tsv", sep="\t")
eval_df = pd.read_csv("data/eval.tsv", sep="\t")
pair_lang = eval_df[["pairid", "lang"]].drop_duplicates()
df = bypair.merge(pair_lang, on="pairid", how="left")

df["model"] = df["model"].str.lower()
df["lang"] = df["lang"].str.lower()

sae_tp_df = df[(df["model"] == "sae") & (df["lang"] == "sae") & (df["acc"] == 1)]
print(sae_tp_df.head())
sae_tp = len(sae_tp_df)
sae_fp = len(df[(df["model"] == "sae") & (df["lang"] == "yoda") & (df["acc"] == 1)])
sae_fn = len(df[(df["model"] == "sae") & (df["lang"] == "sae") & (df["acc"] == 0)])

sae_precision = sae_tp / (sae_tp + sae_fp) if (sae_tp + sae_fp) > 0 else 0
sae_recall = sae_tp / (sae_tp + sae_fn) if (sae_tp + sae_fn) > 0 else 0

yoda_tp_df = df[(df["model"] == "yoda") & (df["lang"] == "yoda") & (df["acc"] == 1)]
print(yoda_tp_df.head())
yoda_tp = len(yoda_tp_df)
yoda_fp = len(df[(df["model"] == "yoda") & (df["lang"] == "sae") & (df["acc"] == 1)])
yoda_fn = len(df[(df["model"] == "yoda") & (df["lang"] == "yoda") & (df["acc"] == 0)])

yoda_precision = yoda_tp / (yoda_tp + yoda_fp) if (yoda_tp + yoda_fp) > 0 else 0
yoda_recall = yoda_tp / (yoda_tp + yoda_fn) if (yoda_tp + yoda_fn) > 0 else 0

print()
print("SAE Model — Precision:", round(sae_precision, 3), "Recall:", round(sae_recall, 3))
print("Yoda Model — Precision:", round(yoda_precision, 3), "Recall:", round(yoda_recall, 3))


    Unnamed: 0 model  pairid  expected  unexpected      diff   acc lang
0            0   sae       0  1.899950    5.492048 -3.592098  True  sae
2            2   sae       2  3.328440    8.439228 -5.110788  True  sae
4            4   sae       4  2.186003    4.821915 -2.635911  True  sae
6            6   sae       6  2.245932    3.139640 -0.893708  True  sae
10          10   sae      10  2.109393    4.988500 -2.879107  True  sae
      Unnamed: 0 model  pairid  expected  unexpected      diff   acc  lang
1985        1985  yoda       1  4.219545    5.721207 -1.501662  True  yoda
1987        1987  yoda       3  3.967229    6.462499 -2.495270  True  yoda
1989        1989  yoda       5  2.308318    5.326268 -3.017950  True  yoda
1991        1991  yoda       7  3.238636    5.515700 -2.277065  True  yoda
1993        1993  yoda       9  2.184380    3.987483 -1.803103  True  yoda

SAE Model — Precision: 0.681 Recall: 0.947
Yoda Model — Precision: 0.689 Recall: 0.984


2. Yes, as both models have a high recall, they did learn different syntactic patterns, consistent with the sentence structures in their training data.

## Part 4 (optional): Implement a bidirectional version of the LSTM_LM 

In [3]:
import MaskedLM

train_data_sae = MaskedLM.MaskedLM_Dataset("data/sae_train.tsv", "data/HW5_vocab.txt", 20)
val_data_sae = MaskedLM.MaskedLM_Dataset("data/sae_val.tsv", "data/HW5_vocab.txt", 20)

model_sae = MaskedLM.LSTM_MLM(train_data_sae.vocabSize, nEmbed, nHidden, nLayers)
trainer = MaskedLM.MaskedLM_Trainer(num_epochs=50, lr=0.5, batch_size=16, device="cpu")

trainer.train(model_sae, train_data_sae, val_data_sae)

Epoch 0:	 Avg Train Loss: 2.49108	 Avg Val Loss: 1.1616
Epoch 10:	 Avg Train Loss: 1.01644	 Avg Val Loss: 0.58069
Epoch 20:	 Avg Train Loss: 0.7173	 Avg Val Loss: 0.81192
Epoch 30:	 Avg Train Loss: 0.45426	 Avg Val Loss: 1.01435
Epoch 40:	 Avg Train Loss: 0.39774	 Avg Val Loss: 1.10101


In [4]:
train_data_yoda = MaskedLM.MaskedLM_Dataset("data/yoda_train.tsv", "data/HW5_vocab.txt", 20)
val_data_yoda   = MaskedLM.MaskedLM_Dataset("data/yoda_val.tsv", "data/HW5_vocab.txt", 20)

model_yoda = MaskedLM.LSTM_MLM(train_data_yoda.vocabSize, nEmbed, nHidden, nLayers)
trainer = MaskedLM.MaskedLM_Trainer(num_epochs=50, lr=0.5, batch_size=16, device="cpu")

trainer.train(model_yoda, train_data_yoda, val_data_yoda)

Epoch 0:	 Avg Train Loss: 2.3946	 Avg Val Loss: 0.74857
Epoch 10:	 Avg Train Loss: 0.95578	 Avg Val Loss: 0.57381
Epoch 20:	 Avg Train Loss: 0.60012	 Avg Val Loss: 0.69642
Epoch 30:	 Avg Train Loss: 0.40103	 Avg Val Loss: 0.96953
Epoch 40:	 Avg Train Loss: 0.36716	 Avg Val Loss: 1.04359


In [5]:
eval_data = MaskedLM.MaskedLM_Dataset("data/eval.tsv", "data/HW5_vocab.txt", 20)

evaluator = MaskedLM.MaskedLM_Evaluator(eval_data, device="cpu")

models = {
    "SAE": model_sae,
    "Yoda": model_yoda
}

evaluator.save_preds(models, "mlm_eval_output.tsv")

In [6]:
import pandas as pd
import matplotlib.pyplot as plt

bypair = pd.read_csv("mlm_eval_results_bypair.tsv", sep="\t")
eval_df = pd.read_csv("data/eval.tsv", sep="\t")
pair_lang = eval_df[["pairid", "lang"]].drop_duplicates()
df = bypair.merge(pair_lang, on="pairid", how="left")

df["model"] = df["model"].str.lower()
df["lang"] = df["lang"].str.lower()

sae_tp_df = df[(df["model"] == "sae") & (df["lang"] == "sae") & (df["acc"] == 1)]
print(sae_tp_df.head())
sae_tp = len(sae_tp_df)
sae_fp = len(df[(df["model"] == "sae") & (df["lang"] == "yoda") & (df["acc"] == 1)])
sae_fn = len(df[(df["model"] == "sae") & (df["lang"] == "sae") & (df["acc"] == 0)])

sae_precision = sae_tp / (sae_tp + sae_fp) if (sae_tp + sae_fp) > 0 else 0
sae_recall = sae_tp / (sae_tp + sae_fn) if (sae_tp + sae_fn) > 0 else 0

yoda_tp_df = df[(df["model"] == "yoda") & (df["lang"] == "yoda") & (df["acc"] == 1)]
print(yoda_tp_df.head())
yoda_tp = len(yoda_tp_df)
yoda_fp = len(df[(df["model"] == "yoda") & (df["lang"] == "sae") & (df["acc"] == 1)])
yoda_fn = len(df[(df["model"] == "yoda") & (df["lang"] == "yoda") & (df["acc"] == 0)])

yoda_precision = yoda_tp / (yoda_tp + yoda_fp) if (yoda_tp + yoda_fp) > 0 else 0
yoda_recall = yoda_tp / (yoda_tp + yoda_fn) if (yoda_tp + yoda_fn) > 0 else 0

print()
print("SAE Model — Precision:", round(sae_precision, 3), "Recall:", round(sae_recall, 3))
print("Yoda Model — Precision:", round(yoda_precision, 3), "Recall:", round(yoda_recall, 3))


    Unnamed: 0 model  pairid   expected  unexpected      diff   acc lang
0            0   sae       0   6.876093   11.433805 -4.557712  True  sae
2            2   sae       2   6.954223   11.508327 -4.554105  True  sae
10          10   sae      10   9.753015   13.339173 -3.586158  True  sae
12          12   sae      12  11.161876   14.149897 -2.988021  True  sae
14          14   sae      14   8.977935   11.095739 -2.117804  True  sae
      Unnamed: 0 model  pairid   expected  unexpected      diff   acc  lang
1985        1985  yoda       1  14.381724   16.705772 -2.324048  True  yoda
1987        1987  yoda       3  13.004164   15.708098 -2.703934  True  yoda
1999        1999  yoda      15  11.059991   12.246239 -1.186248  True  yoda
2011        2011  yoda      27  13.026349   13.529872 -0.503523  True  yoda
2015        2015  yoda      31  10.864851   14.145067 -3.280216  True  yoda

SAE Model — Precision: 0.542 Recall: 0.561
Yoda Model — Precision: 0.593 Recall: 0.525


The causal models achieve much higher recall showing they effectively learned to identify their respective word order patterns. The masked models, in contrast, perform only slightly better than random guessing. This suggests that word order information is primarily captured by sequential dependencies rather than bidirectional context. The causal LSTM learns by predicting the next word given previous words, which directly encodes word order constraints. The masked model, which can see context from both directions, loses this critical sequential information.